In [1]:
!pip install pyspark

## Spark Assignment Answers


Question 1 :-

Driver: The central process that maintains the Spark application session,
converts user code into execution plans/DAGs, schedules tasks, and coordinates execution with worker nodes.

Cluster Manager: The service (e.g., YARN, Standalone, Kubernetes) responsible for allocating CPU, memory, and physical resources across the cluster to run the application.

Executor: Worker processes running on cluster nodes that execute assigned tasks, perform data processing in parallel, and store cached data in memory or disk.


Question 2 :-

Lazy evaluation defers execution until an Action (e.g., .show(), .count(), .write) is called. This enables Spark's Catalyst Optimizer to analyze the entire chain of transformations, optimize the physical execution plan, skip redundant operations, and perform optimization techniques like Predicate Pushdown (filtering data at the file read level).

Question 3 :-

df = spark.read.option("header", "true") \
               .option("inferSchema", "true") \
               .csv("data/source.csv")

Question 4 :-

CSV (Row-based): Stores data line-by-line. Querying specific columns requires reading the entire file, resulting in higher I/O overhead. CSV does not preserve data types or support native compression.

Parquet (Columnar): Stores data grouped by columns. Queries selecting specific columns only read the required column data from disk (Projection Pruning), drastically reducing I/O. Parquet offers high compression ratios, embedded schemas, and optimized read speed for analytical workloads.

Question 5:-

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Q5_Solution").getOrCreate()

# 2. Create sample data for 'df'
data = [
    (101, "Electronics", 299.99),
    (102, "Clothing", 49.99),
    (103, "Electronics", 899.00)
]
columns = ["product_id", "category", "price"]

df = spark.createDataFrame(data, schema=columns)

# 3. Question 5 Query
df_result = df.filter(col("category") == "Electronics") \
              .select("product_id", "price")

# 4. Show Output
df_result.show()

+----------+------+
|product_id| price|
+----------+------+
|       101|299.99|
|       103| 899.0|
+----------+------+



Question 6 :-


In [5]:
from pyspark.sql.functions import col

df_revised = df.withColumnRenamed("old_name", "new_name") \
               .withColumn("price", col("price").cast("double"))

Question 7 :-

Spark builds a Directed Acyclic Graph (DAG) to record every transformation applied to an RDD or DataFrame. If a partition of data is lost due to a worker node failure, Spark does not need to recompute the entire pipeline. Instead, it uses the Lineage Graph to recompute only the lost data partitions from the original source.

Question 8 :-

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Q8_Solution").getOrCreate()

# 2. Define sample dataset for 'df_orders'
data = [
    (1, "Completed", 1200.0),
    (2, "Pending", 1500.0),
    (3, "Completed", 800.0),
    (4, "Completed", 2500.0)
]
columns = ["order_id", "status", "amount"]

df_orders = spark.createDataFrame(data, schema=columns)

# 3. Question 8 Query
df_filtered = df_orders.filter((col("status") == "Completed") & (col("amount") > 1000))

# 4. Show Output
df_filtered.show()

+--------+---------+------+
|order_id|   status|amount|
+--------+---------+------+
|       1|Completed|1200.0|
|       4|Completed|2500.0|
+--------+---------+------+



Question 9 :-

Predicate Pushdown pushes filter conditions (WHERE clauses) down to the storage layer before loading data into Spark memory. Because Parquet files contain metadata (min/max values per block), Spark skips reading file blocks that do not match the filter criteria. This minimizes disk I/O, reduces network traffic, and significantly decreases memory consumption.

Question 10 :-

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Q10_Solution").getOrCreate()

# 2. Define sample dataset with 'base_price' column
data = [
    (101, 100.0),
    (102, 250.5),
    (103, 500.0)
]
columns = ["product_id", "base_price"]

df = spark.createDataFrame(data, schema=columns)

# 3. Question 10 Query
df_final = df.withColumn("final_price", round(col("base_price") * 1.18, 2))

# 4. Show Output
df_final.show()

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|       101|     100.0|      118.0|
|       102|     250.5|     295.59|
|       103|     500.0|      590.0|
+----------+----------+-----------+



Question 11 :-

Transformations: Operations that define a new DataFrame without executing computation immediately (lazy evaluation).

Examples: .filter(), .select(), .withColumnRenamed()

Actions: Operations that trigger execution by computing and returning a result to the driver or writing data to storage.

Examples: .count(), .show(), .collect(), .write()

Question 12 :-

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Q12_Solution").getOrCreate()

# 2. Setup mock data and save it as a Parquet file
data = [(101, "Alice"), (None, "Bob"), (103, "Charlie")]
columns = ["user_id", "name"]
spark.createDataFrame(data, columns).write.mode("overwrite").parquet("sample_input.parquet")

# 3. Question 12 Steps: Load Parquet -> Filter Nulls -> Save to CSV
df_parquet = spark.read.parquet("sample_input.parquet")

df_filtered = df_parquet.filter(col("user_id").isNotNull())

df_filtered.write.mode("overwrite").option("header", "true").csv("sample_output_csv")

# 4. Display result to confirm execution
print("Filtered Data:")
df_filtered.show()

Filtered Data:
+-------+-------+
|user_id|   name|
+-------+-------+
|    101|  Alice|
|    103|Charlie|
+-------+-------+



Question 13 :-

Client Mode: The Spark Driver process runs on the machine where the job was submitted (e.g., local machine/client). It is typically used for interactive exploration and debugging.

Cluster Mode: The Spark Driver process runs inside an executor container on one of the worker nodes within the cluster. It is recommended for production deployments as it keeps driver execution stable inside the cluster environment.

Question 14 :-

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("Q14_Solution").getOrCreate()

# 2. Define sample dataset with 'region' and 'priority' columns
data = [
    (101, "North", "Low"),
    (102, "South", "High"),
    (103, "East", "Low"),
    (104, "West", "Medium")
]
columns = ["id", "region", "priority"]

df = spark.createDataFrame(data, schema=columns)

# 3. Question 14 Query
df_filtered = df.filter((col("region") == "North") | (col("priority") == "High"))

# 4. Show Output
df_filtered.show()

+---+------+--------+
| id|region|priority|
+---+------+--------+
|101| North|     Low|
|102| South|    High|
+---+------+--------+



Question 15 :-

.show(5) fetches only 5 records and prints them in the terminal, consuming minimal driver memory..collect() brings all partitions of data from worker nodes directly into the Driver node's single-machine memory. On multi-terabyte datasets, this causes an OutOfMemory (OOM) error and crashes the Spark application.